# 03 — Korelacje i metryki klastrowania (bez ground-truth)

Odpowiednik `cohort_tests copy` (korelacje, odleglosci), rozszerzony o **metryki
klastrowania**. Wszystko liczone z nauczonej przestrzeni utajonej `Z` i etykiet kohort —
bez prawdziwego `mu`. Pytanie: **czy kohorty separuja sie w przestrzeni, ktorej nauczyl
sie model?**

| Test | Co liczymy |
|------|------------|
| 2 | Korelacje miedzy czynnikami `Z` i miedzy profilami kohort `E[gamma*delta]` |
| 3 | Profil kohortowy `E[gamma]`, `E[gamma*delta]` |
| 4 | Odleglosci L2 miedzy centroidami kohort + dendrogram |
| 5 | Metryki klastrowania: silhouette, Davies-Bouldin, Calinski-Harabasz, within/between + PCA |
| 6 | Jak separacja kohort zalezy od `pi` |

## Kontrakt danych
- `views` (Views z cohorts), `K`. Reszta wyprowadzana automatycznie.

## Komorka 1 — wczytanie danych

In [ ]:
# =====================================================================
#  KOMORKA 1 — WCZYTANIE DANYCH
# =====================================================================
import os, sys, warnings
# Znajdz katalog zawierajacy pakiet `src` (dziala z final_notebooks/ i z roota repo).
_p = os.path.abspath('')
for _ in range(4):
    if os.path.isdir(os.path.join(_p, 'src')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
    _p = os.path.dirname(_p)
warnings.filterwarnings('ignore')

import numpy as np
from src.views import Views


def make_synthetic_cohort_data(C=4, K=4, n_active=2, n_per_cohort=120,
                               dims=(20, 15), sigma=0.6, noise=1.0,
                               mu_scale=1.9, block_sparse_W=False, seed=0):
    """Domyslne dane DEMO (do podmiany na wlasne).

    C kohort o roznych profilach mu_c w przestrzeni K czynnikow; tylko pierwsze
    `n_active` czynnikow rozni kohorty (reszta wspolna -- "cicha"). Widoki powstaja
    jako Y_m = Z W_m^T + szum. Zwraca (views, codes); NIE zwraca ground-truth --
    cala analiza korzysta tylko z `views` i etykiet kohort.
    """
    rng = np.random.default_rng(seed)
    mu = np.zeros((C, K))
    mu[:, :n_active] = rng.normal(scale=mu_scale, size=(C, n_active))
    codes = np.repeat(np.arange(C), n_per_cohort)
    Z = np.array([rng.normal(mu[c], sigma) for c in codes])
    Ys = []
    for d in dims:
        if block_sparse_W:
            W = np.zeros((d, K))                              # prawdziwe zera poza blokiem
            for k, idx in enumerate(np.array_split(np.arange(d), K)):
                W[idx, k] = rng.normal(loc=1.5, scale=0.25, size=len(idx))  # blok cech -> czynnik k
        else:
            W = rng.normal(size=(d, K))
        Ys.append(Z @ W.T + rng.normal(scale=noise, size=(Z.shape[0], d)))
    cohorts = np.array([f'c{c}' for c in codes])
    return Views.from_list(Ys, cohorts=cohorts), codes


# >>>>>>>>>>>>>>>>>>>>>>>  PODMIEN TE SEKCJE NA SWOJE DANE  >>>>>>>>>>>>>>>>>>>>>>>
# Wymagane: `views` (Views z cohorts) oraz `K`.
views, _ = make_synthetic_cohort_data(C=5, K=4, n_active=3, mu_scale=2.0, seed=0)
K = 4
# <<<<<<<<<<<<<<<<<<<<<<<  KONIEC SEKCJI DO PODMIANY  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# Wyprowadzenie etykiet kohort z danych (dziala tez dla Twoich danych):
assert views.cohorts is not None, "views musi miec ustawione cohorts (etykieta kohorty per probka)."
cohort_names, codes = np.unique(views.cohorts, return_inverse=True)
C = len(cohort_names)
factors = [f'Z{k}' for k in range(K)]
print(f"N = {views.N} probek | widoki (liczba cech): {[v.D for v in views.simple]}")
print(f"C = {C} kohort: {list(cohort_names)} | K = {K} czynnikow do dopasowania")

## Setup — model, helpery i metryki klastrowania

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import FACTModel
from src.enums import Likelihood, WPrior, ZPrior
from src.model_config import CohortPriorConfig, ModelConfig, SimpleViewConfig

from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score)
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

def fit_cohort(views, K, pi=0.5, max_iter=100, seed=0):
    """Dopasuj CohortFACTM: WPrior.ARD_SS + ZPrior.COHORT, prior spike-and-slab `pi`."""
    cfg = ModelConfig(
        simple_view_configs=[SimpleViewConfig(likelihood=Likelihood.NORMAL, w_prior=WPrior.ARD_SS)
                             for _ in range(views.num_simple)],
        structured_view_configs=[],
        z_priors=[ZPrior.COHORT] * K,
        cohort_prior_config=CohortPriorConfig(pi=pi),
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m

def cohort_gamma_delta(model, K):
    """Macierze (C, K): E[gamma] (prawdopodobienstwo aktywacji kohortowej) i E[delta]
    (przesuniecie sredniej). Wiersze w kolejnosci np.unique(cohorts) == `cohort_names`."""
    priors = model.fa.node_z.z_priors
    assert priors[0].__class__.__name__ == 'CohortZPrior', "model nie ma cohort-prior"
    E_g = np.column_stack([priors[k].E_gamma for k in range(K)])
    E_d = np.column_stack([priors[k].E_delta for k in range(K)])
    return E_g, E_d

def clustering_metrics(X, labels):
    """Kilka metryk separacji kohort w przestrzeni X.
    silhouette / Calinski-Harabasz: wyzej = lepiej; Davies-Bouldin / within-between: nizej = lepiej."""
    out = {}
    out['silhouette'] = float(silhouette_score(X, labels))
    out['davies_bouldin'] = float(davies_bouldin_score(X, labels))
    out['calinski_harabasz'] = float(calinski_harabasz_score(X, labels))
    uniq = np.unique(labels)
    cents = np.array([X[labels == c].mean(0) for c in uniq])
    within = np.mean([np.linalg.norm(X[labels == c] - cents[i], axis=1).mean()
                      for i, c in enumerate(uniq)])
    bp = cents[:, None, :] - cents[None, :, :]
    between = np.sqrt((bp ** 2).sum(-1))[np.triu_indices(len(cents), 1)].mean()
    out['within/between'] = float(within / between)
    return out

## Test 1 — dopasowanie modelu i ekstrakcja

In [ ]:
m = fit_cohort(views, K=K, pi=0.5, seed=0)
Z_hat = m.get_latent_factors()            # (N, K) nauczona przestrzen utajona
E_g, E_d = cohort_gamma_delta(m, K)       # (C, K)
profile = E_g * E_d                       # profil kohortowy (C, K)
print(f"Z_hat: {Z_hat.shape} | ELBO = {float(m.elbo_sequence[-1]):.1f}")
print("mean E[gamma] per czynnik:", dict(zip(factors, E_g.mean(0).round(3))))

## Test 2 — korelacje miedzy czynnikami i miedzy kohortami

Lewo: macierz korelacji nauczonych czynnikow `Z` (poza-diagonalne ~0 = czynniki
nieskorelowane, dobra faktoryzacja). Prawo: korelacje profili kohortowych
`E[gamma*delta]` (ktore kohorty sa do siebie podobne).

In [ ]:
corrZ = np.corrcoef(Z_hat.T)
corrC = np.corrcoef(profile)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
sns.heatmap(corrZ, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1,
            xticklabels=factors, yticklabels=factors, ax=axes[0])
axes[0].set_title('Korelacje miedzy czynnikami Z (poza-diag. ~0 = dobrze)')
sns.heatmap(corrC, annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1,
            xticklabels=cohort_names, yticklabels=cohort_names, ax=axes[1])
axes[1].set_title('Korelacje profili kohortowych E[gamma*delta]')
plt.tight_layout(); plt.show()

## Test 3 — profil kohortowy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 0.8 + 0.45 * C))
sns.heatmap(E_g, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=factors, yticklabels=cohort_names, ax=axes[0])
axes[0].set_title('E[gamma] (kohorta x czynnik)')
vmax = np.abs(profile).max()
sns.heatmap(profile, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-vmax, vmax=vmax,
            xticklabels=factors, yticklabels=cohort_names, ax=axes[1])
axes[1].set_title('E[gamma*delta] — profil kohortowy')
plt.tight_layout(); plt.show()

## Test 4 — odleglosci miedzy kohortami + dendrogram

Centroidy kohort w nauczonej przestrzeni `Z`, macierz odleglosci L2 oraz hierarchiczne
klastrowanie kohort (ktore kohorty lacza sie najwczesniej).

In [ ]:
cents = np.array([Z_hat[codes == c].mean(0) for c in range(C)])
diff = cents[:, None, :] - cents[None, :, :]
D = np.sqrt((diff ** 2).sum(-1))          # (C, C) symetryczna, zero na diagonali

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
sns.heatmap(D, annot=True, fmt='.2f', cmap='mako',
            xticklabels=cohort_names, yticklabels=cohort_names, ax=axes[0])
axes[0].set_title('Odleglosci L2 miedzy centroidami kohort (w przestrzeni Z)')
Zlink = linkage(squareform(D, checks=False), method='average')
dendrogram(Zlink, labels=list(cohort_names), ax=axes[1])
axes[1].set_title('Hierarchiczne klastrowanie kohort'); axes[1].set_ylabel('odleglosc')
plt.tight_layout(); plt.show()

## Test 5 — metryki klastrowania (kilka)

Separacja kohort w przestrzeni `Z` traktowana jak jakosc klastrowania (etykiety =
kohorty). Kilka metryk plus silhouette per kohorta i projekcja PCA.

In [ ]:
mets = clustering_metrics(Z_hat, codes)
print("Metryki separacji kohort w przestrzeni Z_hat:")
for k, v in mets.items():
    print(f"  {k:18s}: {v:.3f}")

sil_samp = silhouette_samples(Z_hat, codes)
sil_per_cohort = [sil_samp[codes == c].mean() for c in range(C)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(cohort_names, sil_per_cohort, color='teal')
axes[0].axhline(mets['silhouette'], color='gray', ls='--',
                label=f"ogolny silhouette = {mets['silhouette']:.2f}")
axes[0].set_title('Sredni silhouette per kohorta'); axes[0].set_ylabel('silhouette')
axes[0].legend(); axes[0].grid(alpha=0.3, axis='y')

proj = PCA(n_components=2).fit_transform(Z_hat)
for c in range(C):
    msk = codes == c
    axes[1].scatter(proj[msk, 0], proj[msk, 1], s=12, alpha=0.6, label=cohort_names[c])
axes[1].set_title('PCA(Z_hat) — separacja kohort'); axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Test 6 — jak separacja kohort zalezy od `pi`

In [ ]:
pi_grid = [0.1, 0.3, 0.5, 0.7, 0.9]
rows = []
for pi in pi_grid:
    mp = fit_cohort(views, K=K, pi=pi, seed=0)
    rows.append({'pi': pi, **clustering_metrics(mp.get_latent_factors(), codes)})
dfm = pd.DataFrame(rows)
print(dfm.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(dfm['pi'], dfm['silhouette'], 'o-', color='teal')
axes[0].set_title('silhouette vs pi (wyzej = lepiej)')
axes[1].plot(dfm['pi'], dfm['davies_bouldin'], 'o-', color='C3')
axes[1].set_title('Davies-Bouldin vs pi (nizej = lepiej)')
axes[2].plot(dfm['pi'], dfm['calinski_harabasz'], 'o-', color='C2')
axes[2].set_title('Calinski-Harabasz vs pi (wyzej = lepiej)')
for ax in axes:
    ax.set_xlabel('pi'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()